### Parse Script. Original C++ written by Daryl Watson. Translated and heavily updated by Kyle Moore

In [1]:
import csv
import re
import pandas
import random

In [2]:
date_subpatn = '[0-9]{1,2}/[0-9]{1,2}/[0-9]{4}'
date_subpatn_comp = re.compile(date_subpatn)
date_patn = re.compile(f'^[\S\s]*\[{date_subpatn}( - {date_subpatn})?\]$')

ansr_patn = re.compile('^([0-9]{0,3}%|[*])')

citn_patn = re.compile('^Citation: ')

qnid_patn = re.compile('^Question: \[[\w.]+\]')

stid_patn = re.compile('^Study: \[Roper #[0-9.]+\]')

demo_patn = re.compile('^Sample: ')

resn_patn = re.compile('^Sample Size: [0-9]+')

strt_patn = re.compile('^-{60}$')

In [3]:
def parse_date(raw_line):
    return re.findall(date_subpatn_comp, raw_line.strip())[-1]

    # match = re.search(r'\[(.*?)\]', line)
    # if match:
    #     date_range = match.group(1).split('-')
    #     if len(date_range) > 1:
    #         date = date_range[1].strip()
    #     else:
    #         date = date_range[0].strip()
    #     csv_writer.write(f"{date},")

def parse_ques(raw_line):
    return raw_line.strip()

def parse_ansr(raw_lines):
    # Note that answer ratios can be any of: ##%, #%, %, or *. Both % and * should be interpreted as 0 FIXED
    # answer groups can have blank lines :/ FIXED
    # answer groups can have new lines mid answer :/ FIXED

    regrouped_lines = []

    for ln_raw in raw_lines:
        stripped = ln_raw.strip()
        if re.match(ansr_patn, stripped):
            if stripped[0] == '%' or stripped[0] == '*':
                stripped = '0%' + stripped[1:]
            regrouped_lines.append(stripped)
        else:
            regrouped_lines[-1] = (regrouped_lines[-1] + ' ' + ln_raw).strip()

    ratios, answer_txts = [], []
    for ln in regrouped_lines:
        ln_split = ln.split(maxsplit=1)

        answer_txts.append(ln_split[1])
        ratios.append(int(ln_split[0][:-1]))
        
    return answer_txts, ratios

def parse_qnid(raw_line):
    stripped = raw_line.strip()
    if not stripped.startswith('Question: [') or not stripped.endswith(']'):
        raise ValueError(f'Error: could not parse QNID. stripped raw line - {stripped}')
    return stripped[11:-1]

def parse_stid(raw_line):
    stripped = raw_line.strip()
    if not stripped.startswith('Study: [Roper #') or not stripped.endswith(']'):
        raise ValueError(f'Error: could not parse STID. stripped raw line - {stripped}')
    return stripped[15:-1]

def parse_citn(raw_line):
    stripped = raw_line.strip()
    if not stripped.startswith('Citation: '):
        raise ValueError(f'Error: could not parse CITN. stripped raw line - {stripped}')
    return stripped[10:].split('.')[0]

def parse_demo(raw_line):
    stripped = raw_line.strip()
    if not stripped.startswith('Sample: '):
        raise ValueError(f'Error: could not parse DEMO. stripped raw line - {stripped}')
    return stripped[8:]

def parse_resn(raw_line):
    stripped = raw_line.strip()
    if not stripped.startswith('Sample Size: '):
        raise ValueError(f'Error: could not parse RESN. stripped raw line - {stripped}')
    smp_size = stripped[13:]
    try:
        smp_size = int(smp_size)
    except:
        raise ValueError(f'Error: cound not convert sample count to numeric. As str - {smp_size}')
    return smp_size


In [4]:
# All phrases found in dataset that indicate clearly that some subset
# are non-responses. Explicitly does not omit choices that are non-commital
# responses (e.g. 'unsure' or 'don\'t know')
refusal_phrases = [
    'don\'t know/refused',
    'don\'t know/skipped',
    'don\'t know/skippedrefused',
    'no answer',
    'not selected',
    'not selected/no answer',
    'not sure/refused',
    'not sure/skipped',
    'omit',
    'refused',
    'refused/web blank',
    'skip',
    'skipped',
    'skipped on web',
    'skipped/refused',
    'skipped/web blank',
    'web blank',
]

In [5]:
def parse(fn, shuffle_answers=True):
    field_names = [
        'Date',
        'Question',
        'Answers',
        'Answer_Ratios',
        'Citation',
        'Question_Id',
        'Study_Id',
        'Demographic',
        'Response_Count',
    ]

    

    data_dict = {field_nm : [] for field_nm in field_names}

    excluded_ref_len = 0
    excluded_ans_count = 0

    with open(fn, encoding='utf-8') as data_file:
        lines = data_file.readlines()
        num_lines = len(lines)
        pointer = 2 # skip first two lines, always iRoper metaheader and blank line

        while pointer < num_lines:
            while not re.match(date_patn, lines[pointer]):
                pointer += 1
            date_loc = pointer
            
            pointer += 2
            ques_loc = pointer
            if lines[ques_loc].strip() == '':
                raise ValueError(f'Unexpected format(ANSR) at line {date_loc+1}.\n{lines[date_loc]}')

            while not re.match(ansr_patn, lines[pointer]) and not re.match(strt_patn, lines[pointer]):
                if re.match(strt_patn, lines[pointer]):
                    raise ValueError(f'Unexpected next question at line {pointer}. Expected: ANSR')
                pointer += 1
            ansr_loc = pointer

            while not re.match(citn_patn, lines[pointer]) and not re.match(strt_patn, lines[pointer]):
                if re.match(strt_patn, lines[pointer]):
                    raise ValueError(f'Unexpected next question at line {pointer}. Expected: CITN')
                pointer += 1
            citn_loc = pointer

            while not re.match(qnid_patn, lines[pointer]) and not re.match(strt_patn, lines[pointer]):
                if re.match(strt_patn, lines[pointer]):
                    raise ValueError(f'Unexpected next question at line {pointer}. Expected: QNID')
                pointer += 1
            qnid_loc = pointer

            while not re.match(stid_patn, lines[pointer]) and not re.match(strt_patn, lines[pointer]):
                if re.match(strt_patn, lines[pointer]):
                    raise ValueError(f'Unexpected next question at line {pointer}. Expected: STID')
                pointer += 1
            stid_loc = pointer

            while not re.match(demo_patn, lines[pointer]) and not re.match(strt_patn, lines[pointer]):
                if re.match(strt_patn, lines[pointer]):
                    raise ValueError(f'Unexpected next question at line {pointer}. Expected: DEMO')
                pointer += 1
            demo_loc = pointer

            while not re.match(resn_patn, lines[pointer]) and not re.match(strt_patn, lines[pointer]):
                if re.match(strt_patn, lines[pointer]):
                    raise ValueError(f'Unexpected next question at line {pointer}. Expected: RESN')
                pointer += 1
            resn_loc = pointer

            while pointer < num_lines and not re.match(strt_patn, lines[pointer]):
                pointer += 1

            # Extract values
            date = parse_date(lines[date_loc])
            ques = parse_ques(lines[ques_loc])
            ansr, rati = parse_ansr(lines[ansr_loc:citn_loc-1])
            citn = parse_citn(lines[citn_loc])
            qnid = parse_qnid(lines[qnid_loc])
            stid = parse_stid(lines[stid_loc])
            demo = parse_demo(lines[demo_loc])
            resn = parse_resn(lines[resn_loc])

            # Rough way to exclude multi-select questions where possible. Total should add up to
            # 100, with some allowance for over/under summing from rounding errors. Rounding errors
            # can never explain vatiation from 100 greater than the number of choices. Will likely
            # have missed some questions where the total adds up to ~100 by happenstance. This is
            # likely a necessary evil and unfixable. Should be safe to ignore.
            if sum(rati) <= 100 - len(rati) or sum(rati) >= 100 + len(rati):
                excluded_ans_count += 1
                continue

            # Additional step: remove answer choices that are identifiably just some variation of
            # refused or failed to answer. Also remove questions with only one or no available
            # answer (including ones that had more than 1 answers before filtering).
            filt_ansr, filt_rati = [], []
            for a, r in zip(ansr, rati):
                if a.lower() not in refusal_phrases:
                    filt_ansr.append(a)
                    filt_rati.append(r)
            ansr, rati = filt_ansr, filt_rati
            
            if len(ansr) <= 1:
                excluded_ref_len += 1
                continue

            if shuffle_answers:
                # Shuffle answers to control for positional biases across questions with similar answer options
                ansr, rati = zip(*random.sample(list(zip(ansr, rati)), k=len(ansr)))
                ansr, rati = list(ansr), list(rati)

            data_dict['Date'].append(date)
            data_dict['Question'].append(ques)
            data_dict['Answers'].append(ansr)
            data_dict['Answer_Ratios'].append(rati)
            data_dict['Citation'].append(citn)
            data_dict['Question_Id'].append(qnid)
            data_dict['Study_Id'].append(stid)
            data_dict['Demographic'].append(demo)
            data_dict['Response_Count'].append(resn)

    print(f'{excluded_ans_count} rows removed for invalid answer ratios')
    print(f'{excluded_ref_len} rows removed for too few non-refusal options')


    df = pandas.DataFrame.from_dict(data_dict)
    return df


In [6]:
input_filename = 'iroper_dl.txt'
# input_filename = 'iroper_dl_trunc.txt'

output_fn_csv = 'iroper_data.csv'
output_fn_pkl = 'iroper_data.pkl'

sample = True
sample_n = 3000
shuffle_answers = True

df = parse(input_filename, shuffle_answers=shuffle_answers)

if sample:
    df = df.sample(sample_n, ignore_index=True)

# df.to_csv(output_fn_csv)
# df.to_pickle(output_fn_pkl)


572 rows removed for invalid answer ratios
332 rows removed for too few non-refusal options


In [15]:
df

,Date,Question,Answers,Answer_Ratios,Citation,Question_Id,Study_Id,Demographic,Response_Count
0,10/2/2019,If you felt you were experiencing an issue wit...,"[Somewhat comfortable, Very uncomfortable, Don...","[20, 1, 1, 72, 5, 1]",CBS News,31116850.00025,31116850,National adult,1292
1,04/1/2019,How much confidence do you have in the clergy ...,"[No confidence at all, Not much confidence, So...","[16, 33, 36, 13]",Pew Research Center for the People & the Press,31116476.00046,31116476,National adult,6364
2,04/17/2020,(Since the recent coronavirus (COVID-19 virus)...,"[Once a month, Once a week, More than once a m...","[17, 9, 13, 2, 32, 25]",AARP,31119428.00020,31119428,National adults age 50+,1101
3,11/11/2019,When you talk with friends and family about po...,"[Listen to the conversation more than lead, Ne...","[55, 21, 23]",Pew Research Center for the People & the Press,31117053.00091,31117053,National adult,12043
4,12/11/2017,Have you personally been the victim of sexual ...,"[Yes, Don't know, No]","[21, 0, 78]",Associated Press-NORC Center,USAP.122217N.R20,31114782,National adult,1020
...,...,...,...,...,...,...,...,...,...
2995,04/18/2017,Do you personally own any guns (not including ...,"[No, I don't own any guns, Yes, I own a gun]","[69, 30]",Pew Research Center,31116860.00017,31116860,National adult,4168
2996,09/26/2021,"(Which of the following, if any, do you believ...","[No, I do not believe in this, Yes, I believe ...","[65, 33]",Pew Research Center,31118724.00054,31118724,National adult,6485
2997,10/30/2017,"In general, do you think the Democratic Party ...","[Rich, Don't know/No answer, Poor, Middle clas...","[27, 8, 18, 19, 29]",CBS News,USCBS.110117.R08,31114635,National adult,1109
2998,12/1/2022,Do you support or oppose laws prohibiting disc...,"[Support, Oppose]","[81, 19]",KFF/Washington Post,31120179.00183,31120179,National adult including an oversample of 515 ...,1338
